# When near stops meaning near
Lecture 4 · the curse of dimensionality. **Predict before every run.** Write your guess down.

In [ ]:
# L4.00 Setup
import numpy as np
import matplotlib.pyplot as plt
from math import gamma, pi
from scipy.spatial.distance import pdist

rng = np.random.default_rng(0)

try:
    import ipywidgets as w
    def play(f, **s):
        ctl = {}
        for k, v in s.items():
            if isinstance(v, list):
                ctl[k] = w.Dropdown(options=v)
            elif isinstance(v[3], float):
                ctl[k] = w.FloatSlider(min=v[0], max=v[1], step=v[2], value=v[3])
            else:
                ctl[k] = w.IntSlider(min=v[0], max=v[1], step=v[2], value=v[3])
        w.interact(f, **ctl)
except ImportError:  # no sliders: run once with the defaults
    def play(f, **s):
        f(**{k: (v[0] if isinstance(v, list) else v[3]) for k, v in s.items()})

def ball_share(n):
    # share of the cube [-1, 1]^n filled by the unit ball
    return pi ** (n / 2) / gamma(n / 2 + 1) / 2 ** n

## 1 · Nobody is average
**Predict:** 4,063 US Air Force pilots, 10 body measurements (Daniels, 1952). How many are average (within ±0.3 SD) on all 10?

In [ ]:
# L4.01 The average pilot
daniels = [4063, 1055, 302, 143, 73, 28, 12, 6, 3, 2, 0]  # Daniels 1952, real counts

def pilots(correlation=0.8):
    shared = rng.standard_normal((4063, 1))
    X = np.sqrt(correlation) * shared + np.sqrt(1 - correlation) * rng.standard_normal((4063, 10))
    left = [4063] + list(np.cumprod(np.abs(X) < 0.3, axis=1).sum(0))
    plt.figure(figsize=(7, 3.5))
    plt.plot(range(11), daniels, 'ko-', label='Daniels 1952, real pilots')
    plt.plot(range(11), left, 's--', label=f'simulated, correlation {correlation:.2f}')
    plt.yscale('symlog', linthresh=1)
    plt.xlabel('measurements checked')
    plt.ylabel('pilots still average')
    plt.legend()
    plt.show()

play(pilots, correlation=(0.0, 0.95, 0.05, 0.8))

**Try:** correlation 0. Then: your patients have 60 features.

## 2 · The ball in the box
**Predict:** what share of a cube does the biggest ball inside it fill? In 2D? In 10D?

In [ ]:
# L4.02 Darts in a cube
def darts(n=2):
    X = rng.uniform(-1, 1, (4000, n))
    inside = (X ** 2).sum(1) <= 1
    y = X[:, 1] if n > 1 else rng.uniform(-1, 1, 4000)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
    a.scatter(X[:, 0], y, s=3, c=np.where(inside, 'tab:blue', 'tab:red'))
    a.add_patch(plt.Circle((0, 0), 1, fill=False))
    a.set_aspect('equal')
    a.set_title(f'n = {n}: {inside.mean():.1%} of darts inside (exact {ball_share(n):.2%})')
    dims = np.arange(1, 21)
    b.semilogy(dims, [ball_share(d) for d in dims], 'o-')
    b.semilogy(n, ball_share(n), 'o', ms=12, mfc='none', mec='tab:red')
    b.set_xlabel('dimensions')
    b.set_ylabel('ball ÷ cube')
    plt.tight_layout()
    plt.show()

play(darts, n=(1, 20, 1, 2))

**Predict:** the unit ball (radius 1). In which dimension is its volume largest?

In [ ]:
# L4.03 The biggest ball
dims = np.arange(1, 31)
volume = np.array([pi ** (d / 2) / gamma(d / 2 + 1) for d in dims])
plt.figure(figsize=(7, 3))
plt.bar(dims, volume)
plt.xlabel('dimensions')
plt.ylabel('volume of the unit ball')
plt.title(f'Largest at n = {dims[volume.argmax()]}, then it shrinks to zero')
plt.show()

## 3 · Spheres in the corners
**Predict:** a box of side 4 holds 2ⁿ unit spheres, one in each corner. A sphere in the middle just touches them all. In which dimension does the middle sphere poke out of the box?

In [ ]:
# L4.04 The middle sphere
def corners(n=2):
    r = np.sqrt(n) - 1  # a corner sphere's centre is sqrt(n) away; subtract its radius 1
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
    for cx in (-1, 1):
        for cy in (-1, 1):
            a.add_patch(plt.Circle((cx, cy), 1, fill=False, color='grey'))
    a.add_patch(plt.Circle((0, 0), r, color='tab:red', alpha=0.5))
    a.add_patch(plt.Rectangle((-2, -2), 4, 4, fill=False, lw=2))
    a.set_xlim(-3.5, 3.5)
    a.set_ylim(-3.5, 3.5)
    a.set_aspect('equal')
    a.set_title(f'n = {n}: middle sphere radius {r:.2f}')
    dims = np.arange(1, 21)
    b.plot(dims, np.sqrt(dims) - 1, 'o-', label='middle sphere radius')
    b.axhline(2, color='k', ls='--', label='box wall')
    b.set_xlabel('dimensions')
    b.legend()
    plt.tight_layout()
    plt.show()

play(corners, n=(2, 20, 1, 2))

## 4 · All peel, no orange
**Predict:** in a 60-dimensional orange, what share of the volume is the outer 5% (the peel)?

In [ ]:
# L4.05 The peel
def peel(n=3, thickness=0.05):
    radius = rng.uniform(size=5000) ** (1 / n)  # distance from the centre, uniform in the ball
    plt.figure(figsize=(7, 3))
    plt.hist(radius, bins=50, range=(0, 1))
    plt.axvspan(1 - thickness, 1, color='tab:orange', alpha=0.3)
    plt.xlabel('distance from the centre')
    plt.title(f'n = {n}: {1 - (1 - thickness) ** n:.0%} of the volume is peel')
    plt.show()

play(peel, n=(1, 200, 1, 3), thickness=(0.01, 0.3, 0.01, 0.05))

## 5 · The soap bubble
**Predict:** 1,000 patients, 100 standardised features each. How many are closer than 5 to the average patient?

In [ ]:
# L4.06 Distance to the average patient
def bubble(n=100):
    X = rng.standard_normal((1000, n))
    dist = np.linalg.norm(X, axis=1)  # distance to the mean, which is the origin
    plt.figure(figsize=(7, 3))
    plt.hist(dist, bins=50)
    plt.axvline(np.sqrt(n), color='k', ls='--', label='√n')
    plt.xlim(0, None)
    plt.xlabel('distance to the average patient')
    plt.title(f'n = {n}: closest patient at {dist.min():.1f}; {np.sum(dist < 5)} within 5')
    plt.legend()
    plt.show()

play(bubble, n=[100, 1, 2, 10, 1000])

## 6 · Every pair the same distance
**Predict:** sketch the histogram of all pairwise distances for n = 2, then for n = 1000.

In [ ]:
# L4.07 Pairwise distances
def pairs(n=2, metric='euclidean'):
    d = pdist(rng.random((300, n)), metric=metric)
    plt.figure(figsize=(7, 3))
    plt.hist(d / d.mean(), bins=60, range=(0, 2))
    plt.xlabel('distance ÷ mean distance')
    plt.title(f'n = {n}, {metric}: mean {d.mean():.2f}, SD {d.std():.3f}, max/min {d.max() / d.min():.1f}')
    plt.show()

play(pairs, n=[2, 1, 3, 10, 100, 1000], metric=['euclidean', 'cityblock', 'chebyshev'])

**Look:** the mean grows, the SD does not.

## 7 · Nearest versus farthest
**Predict:** in 2D the farthest neighbour is tens of times farther than the nearest. In 1000D?

In [ ]:
# L4.08 Contrast
dims = [1, 2, 3, 5, 10, 20, 60, 100, 200, 500, 1000]
plt.figure(figsize=(7, 3.5))
for metric, p in [('L1', 1), ('L2', 2), ('L∞', np.inf)]:
    ratio = []
    for n in dims:
        X, q = rng.random((500, n)), rng.random(n)
        d = np.linalg.norm(X - q, ord=p, axis=1)
        ratio.append(d.max() / d.min())
    plt.loglog(dims, ratio, 'o-', label=metric)
    print(f'{metric:3s}', '  '.join(f'n={n}: {r:.2f}' for n, r in zip(dims, ratio) if n in (2, 60, 1000)))
plt.axvline(60, color='grey', ls=':')
plt.xlabel('dimensions')
plt.ylabel('farthest ÷ nearest')
plt.legend()
plt.show()

## 8 · Everyone at right angles
**Predict:** two random patients' feature vectors in 1000D. What is the angle between them?

In [ ]:
# L4.09 Angles
plt.figure(figsize=(7, 3))
for n in [2, 10, 100, 1000]:
    A, B = rng.standard_normal((2, 3000, n))
    cos = (A * B).sum(1) / np.linalg.norm(A, axis=1) / np.linalg.norm(B, axis=1)
    plt.hist(np.degrees(np.arccos(cos)), bins=90, range=(0, 180), histtype='step', lw=2, label=f'n = {n}')
plt.xlabel('angle between two random vectors (degrees)')
plt.legend()
plt.show()

## 9 · Your neighbourhood is not local
**Predict:** 10 features. To catch the 10% most similar patients in a box, how much of each feature's range must the box cover?

In [ ]:
# L4.10 Neighbourhood size
def neighbourhood(share=0.1):
    p = np.arange(1, 61)
    edge = share ** (1 / p)  # Hastie, Tibshirani & Friedman, ESL 2.5
    plt.figure(figsize=(7, 3))
    plt.plot(p, edge)
    plt.ylim(0, 1.05)
    plt.xlabel('features')
    plt.ylabel('share of each range covered')
    plt.title(f'Catch {share:.0%} of patients: {edge[9]:.0%} of every range at 10 features, {edge[59]:.0%} at 60')
    plt.show()

play(neighbourhood, share=(0.01, 0.5, 0.01, 0.1))

## 10 · Structure saves you, noise does not
**Predict:** 60 features driven by 2 hidden factors. Add 500 noise features. Do neighbours survive?

In [ ]:
# L4.11 Hidden factors plus noise
def nearest(M):
    D = np.linalg.norm(M[:, None] - M[None], axis=2)
    np.fill_diagonal(D, np.inf)
    return D.argmin(1)

def structure(factors=2, noise_features=0):
    Z = rng.standard_normal((240, factors))
    S = Z @ rng.standard_normal((factors, 60)) + 0.1 * rng.standard_normal((240, 60))
    S = (S - S.mean(0)) / S.std(0)
    X = np.hstack([S, rng.standard_normal((240, noise_features))])
    kept = np.mean(nearest(S) == nearest(X))
    d = pdist(X)
    plt.figure(figsize=(7, 3))
    plt.hist(d / d.mean(), bins=60, range=(0, 2))
    plt.xlabel('distance ÷ mean distance')
    plt.title(f'{X.shape[1]} columns: SD/mean {d.std() / d.mean():.3f}; nearest neighbour kept {kept:.0%}')
    plt.show()

play(structure, factors=(1, 20, 1, 2), noise_features=(0, 1000, 50, 0))

## 11 · Your data
Needs `features_df_trial.csv` from tutorial 2.

In [ ]:
# L4.12 The course dataset
import os
import pandas as pd

path = '/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/features_df_trial.csv'
if os.path.exists(path):
    df = pd.read_csv(path)
    X = df.drop(columns=['subject', 'activity', 'trial']).select_dtypes('number').dropna(axis=1).values
    X = (X - X.mean(0)) / (X.std(0) + 1e-12)
    for extra in [0, 100, 300, 1000]:
        Xn = np.hstack([X, rng.standard_normal((len(X), extra))])
        d = pdist(Xn)
        print(f'{Xn.shape[1]:5d} columns: SD/mean {d.std() / d.mean():.3f}, max/min {d.max() / d.min():8.1f}')
else:
    print('Mount Drive and run tutorial 2 first:', path)

## Exit check
1. What share of a 6-dimensional cube does its inscribed ball fill?
2. In 60D, what share of a ball's volume lies within 5% of its surface?
3. Your nearest neighbour is at 9.5. The mean distance is 10, the SD 0.24. Close?
4. A colleague adds 300 features: "k-NN will ignore the useless ones." Prediction, and how would you check?